# 📝 의료 데이터 하이브리드 검색 과제 LV1(기초)

<img src="images/09_drugs_loaded_graph.png" width="1400">

교안 01의 흐름을 **단위프로젝트2_개편 버전 C의 e약은요 제품 문서**에 적용합니다.

**데이터**: 프로젝트 전체 519개 문서 중 과제용 14개 제품 문서·43개 청크를 사용합니다. 개체 270개·도메인 관계 459개에 원문·청크를 연결하면 그림의 327노드·516관계가 됩니다. Drug 45개 중 문서가 있는 제품은 14개이며 나머지는 상호작용 대상으로 언급된 약품입니다.

**풀이 방법**: 준비 셀부터 순서대로 실행하세요. 구분선 안의 `[작성]` 부분에서 핵심 코드를 완성합니다. 대입문은 변수명과 `= ...`만 제공하고 우변 전체를 직접 작성합니다. `...`는 호출식이나 여러 줄 코드로 바꿀 수 있습니다. 함수 틀·질문·반복 처리·결과 표와 출력은 제공됩니다. 앞 문항의 결과를 다음 문항에서 이어 쓰고, 자가채점 후에는 원문과 관계 근거도 읽으세요.

**관계 의미**: TREATS는 효능, HAS_SIDE_EFFECT는 이상반응, CAUTION_FOR는 주의 대상의 원문 기록입니다. 이 과제는 제품 문서 검색이며 복용 추천이 아닙니다. 같은 증상을 효능과 이상반응에 함께 언급할 수 있으므로 관계 종류를 구분합니다.

**모델 호출**: 한글 질문 기준 질문 임베딩 1회·Jev 언어 판별 1회입니다. 영어 질문일 때만 한국어 검색어 생성 1회가 추가됩니다. 문서 벡터는 배포 파일을 재사용합니다.


## 준비와 데이터 연결

README의 패키지·환경변수를 설정합니다. 영화 교안을 적재한 실습 DB를 그대로 사용할 수 있습니다. 아래 준비 코드는 의료 그래프를 생성·재사용하며, 검색 문제는 직접 작성합니다.


In [ ]:
# 검색·언어 판별·구조화 출력에 교안과 같은 라이브러리를 사용합니다.
import json
import os
import re
from pathlib import Path
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from pydantic import BaseModel, Field
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem


In [ ]:
# 교안의 .env로 DB와 질문 임베딩 모델을 연결합니다.
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 질문을 자동으로 나누지 않습니다.
)

# 검색어·Cypher·답변에는 같은 모델을 사용합니다.
llm = ChatOpenAI(model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"))


def run_cypher(query, **params):
    """파라미터 쿼리의 결과를 사전 목록으로 반환합니다."""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


원문과 청크·벡터를 읽습니다. `drugs["relations"]`는 claim_id·subject_id·object_id·relation·evidence·source_doc_id·source_url을 가진 관계 목록입니다. `documents_by_id`는 원문 ID로 제목·본문·URL을, `nodes_by_id`는 개체 ID로 name·entity_type을 찾는 사전입니다.


In [ ]:
# 같은 프로젝트에서 검증한 문서·청크와 배포 벡터를 읽습니다.
import sys
sys.path.insert(0, str(material_dir.resolve()))
from graph_data import load_graph, read_vectors

drugs = load_graph(material_dir / "data/drugs_extraction_packet.json")
documents_by_id = drugs["documents"]
nodes_by_id = {node["standard_id"]: node for node in drugs["nodes"]}
chunk_ids, vectors = read_vectors(material_dir / "data/embeddings/drugs.npz", drugs["chunks"], embedding_model)
print("제품 문서:", len(documents_by_id), "청크:", len(chunk_ids))
display(pd.DataFrame([{"source_id": key, "title": doc["title"]} for key, doc in documents_by_id.items()]))


### 의료 그래프를 준비합니다

`DrugEntity`는 원본의 다섯 개체 유형에 덧붙인 공통 ID 레이블입니다. `DrugRankNode`는 이 과제의 Drug·Symptom에만 붙여 PageRank 계산 범위를 지정합니다. 원문은 `DrugDocument`, 청크는 `DrugChunk`에 적재합니다. 이 레이블과 `drug_chunks`·`drug_fulltext` 인덱스는 영화 교안의 검색 대상과 구분됩니다. 같은 이름이나 같은 검색 대상의 기존 인덱스를 정리하고 다시 만들어 이름·설정 충돌을 막습니다. 노드와 저장된 벡터는 유지합니다.


In [ ]:
# [제공 코드] 적재는 교안의 준비 단계를 재사용합니다. 검색 로직은 아래에서 직접 작성합니다.
# DrugEntity는 이 과제 개체의 공통 ID 제약을 위한 추가 레이블입니다.
run_cypher("CREATE CONSTRAINT drug_entity_id IF NOT EXISTS FOR (n:DrugEntity) REQUIRE n.standard_id IS UNIQUE")
run_cypher("CREATE CONSTRAINT drug_document_id IF NOT EXISTS FOR (n:DrugDocument) REQUIRE n.id IS UNIQUE")
run_cypher("CREATE CONSTRAINT drug_chunk_id IF NOT EXISTS FOR (n:DrugChunk) REQUIRE n.id IS UNIQUE")

# 원본 식별자는 파일에 보존하고 DB ID에는 과제 접두어를 붙입니다.
for entity_type in ["Drug", "Symptom", "RiskGroup", "Ingredient", "Manufacturer"]:
    rows = [node for node in drugs["nodes"] if node["entity_type"] == entity_type]
    rank_label = ":DrugRankNode" if entity_type in ["Drug", "Symptom"] else ""
    run_cypher(f"""
        UNWIND $rows AS item
        MERGE (n:DrugEntity {{standard_id: 'day49:' + item.standard_id}})
        SET n:{entity_type}{rank_label}, n.name = item.name, n.aliases = item.aliases
    """, rows=rows)

# 같은 claim_id를 재사용해 중복 적재를 막고 원문 근거를 보존합니다.
for relation_type in ["TREATS", "HAS_SIDE_EFFECT", "CAUTION_FOR", "CONTAINS", "INTERACTS_WITH", "MADE_BY"]:
    rows = [row for row in drugs["relations"] if row["relation"] == relation_type]
    run_cypher(f"""
        UNWIND $rows AS item
        MATCH (a:DrugEntity {{standard_id: 'day49:' + item.subject_id}})
        MATCH (b:DrugEntity {{standard_id: 'day49:' + item.object_id}})
        MERGE (a)-[r:{relation_type} {{claim_id: item.claim_id}}]->(b)
        SET r.evidence = item.evidence, r.source_url = item.source_url,
            r.source_doc_id = item.source_doc_id
    """, rows=rows)


In [ ]:
# [제공 코드] 원문은 문서 제품에만 연결합니다. 상호작용 대상으로 나온 약명에는 문서가 없습니다.
run_cypher("""
    UNWIND $rows AS item
    MERGE (d:DrugDocument {id: item.doc_id})
    SET d.title = item.title, d.text = item.text, d.url = item.url
    WITH d, item
    MATCH (p:DrugEntity:Drug {standard_id: 'day49:rag:drugs:drug:' + item.doc_id})
    MERGE (d)-[:ABOUT_DRUG]->(p)
""", rows=[{"doc_id": key, **doc} for key, doc in documents_by_id.items()])

# 검사한 벡터 행 순서에 맞춰 청크를 저장합니다.
rows = [{"id": chunk_ids[i], "source_id": chunk.metadata["source_doc_id"],
         "text": chunk.page_content, "embedding": vectors[i].tolist()}
        for i, chunk in enumerate(drugs["chunks"])]
run_cypher("""
    UNWIND $rows AS item
    MATCH (d:DrugDocument {id: item.source_id})
    MERGE (c:DrugChunk {id: item.id})
    SET c.text = item.text, c.source_id = item.source_id, c.embedding = item.embedding
    MERGE (c)-[:FROM_DOCUMENT]->(d)
""", rows=rows)


# 같은 이름의 옛 설정과 같은 검색 대상의 다른 이름 인덱스를 정리합니다.
# IF NOT EXISTS만 쓰면 다른 이름의 인덱스 때문에 필요한 이름이 생성되지 않을 수 있습니다.
indexes = run_cypher("""
    SHOW INDEXES YIELD name, type, entityType, labelsOrTypes, properties
    WHERE name IN ['drug_chunks', 'drug_fulltext']
       OR (entityType = 'NODE' AND labelsOrTypes = ['DrugChunk']
           AND ((type = 'VECTOR' AND properties = ['embedding'])
             OR (type = 'FULLTEXT' AND properties = ['text'])))
    RETURN name
""")
for index in indexes:
    name = index["name"].replace("`", "``")
    run_cypher(f"DROP INDEX `{name}` IF EXISTS")
# 아래에서 정해진 이름·차원·분석기로 다시 만듭니다. 노드와 저장된 벡터는 유지됩니다.

# 의료 전용 레이블을 색인해 교안의 영화 검색과 구분합니다.
run_cypher("""
    CREATE VECTOR INDEX drug_chunks IF NOT EXISTS FOR (c:DrugChunk) ON c.embedding
    OPTIONS {indexConfig: {`vector.dimensions`: 768, `vector.similarity_function`: 'cosine'}}
""")
run_cypher("""
    CREATE FULLTEXT INDEX drug_fulltext IF NOT EXISTS FOR (c:DrugChunk) ON EACH [c.text]
    OPTIONS {indexConfig: {`fulltext.analyzer`: 'standard-no-stop-words', `fulltext.eventually_consistent`: false}}
""")
run_cypher("CALL db.awaitIndex('drug_chunks', 300)")
run_cypher("CALL db.awaitIndex('drug_fulltext', 300)")
print("의료 그래프와 두 검색 인덱스 준비 완료")


## 1. 벡터 검색기를 만들고 청크를 찾습니다

**배경**: 효능 설명으로 기준 제품을 찾은 뒤 공통 효능 증상의 다른 제품까지 탐색합니다.

**제공 코드**: formatter 전체·질문 문장·검색 실행 순서·청크 수 출력.

**요구사항**:

- <strong><code>vector_retriever</code></strong>는 VectorRetriever입니다. driver를 연결하고 index_name="drug_chunks", embedder=embedding_model, result_formatter=format_vector로 생성하세요.
- <strong><code>query_vector</code></strong>는 embedding_model.embed_query에 semantic_query를 전달해 한 번 만든 질문 벡터입니다.
- <strong><code>vector_result</code></strong>에는 vector_retriever.search에 query_vector와 top_k=12를 전달한 반환 객체를 담으세요.

**확인 기준**: 질문 벡터는 768차원이며 청크마다 본문·청크 ID·원문 ID·점수가 있습니다.


In [ ]:
def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # 잘못된 인덱스나 이전 검색 결과를 사용하면 복구 순서를 안내합니다.
    if node["source_id"] not in documents_by_id:
        raise ValueError('검색 결과의 원문 ID가 현재 자료에 없습니다. 커널을 재시작하고 적재·검색기 생성·검색 셀을 순서대로 다시 실행하세요.')
    # content는 청크 본문, metadata는 청크 ID·원문 ID·유사도를 담는 사전입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
# ====================================================================
# [작성]
vector_retriever = ...
# ====================================================================

# 이후 검색에서도 같은 질문 벡터를 재사용합니다.
question = "위산과다 속쓰림 신트림이 효능에 적힌 제품 문서를 찾고, 해당 제품과 효능 증상을 공유하는 다른 제품 문서도 찾아 주세요."
semantic_query = "위산과다 속쓰림 신트림"
# ====================================================================
# [작성]
query_vector = ...
# ====================================================================
# ====================================================================
# [작성]
vector_result = ...
# ====================================================================
print("검색 청크 수:", len(vector_result.items))


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(vector_retriever, VectorRetriever), '1번 벡터 검색기를 만들고 청크를 찾습니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert len(query_vector) == 768 and 0 < len(vector_result.items) <= 12, '1번 벡터 검색기를 만들고 청크를 찾습니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(item.content and {"chunk_id", "source_id", "score"} <= item.metadata.keys() for item in vector_result.items), '1번 벡터 검색기를 만들고 청크를 찾습니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 1번 핵심 확인 완료!")


## 2. 청크 결과를 원문별로 합칩니다

**배경**: 한 제품의 여러 청크가 검색되어도 문서 후보는 한 건으로 표시합니다.

**제공 코드**: 함수 틀, rows·seen 초기화, 검색 결과 순회, 반환문, 함수 호출과 제목·점수 표 출력.

**요구사항**:

- <strong><code>result</code></strong>에는 vector_retriever.search로 함수 인자의 query_vector·top_k를 전달한 결과를 담으세요.
- <strong><code>seen</code></strong>에 row["source_id"]가 없을 때만 해당 ID를 추가하고, dict(row)로 만든 metadata 사본을 <strong><code>rows</code></strong>에 추가하세요. 조건문과 두 갱신문을 작성합니다.

**확인 기준**: 원문 ID가 중복되지 않고 각 원문의 최고 점수 청크를 유지합니다.


In [ ]:
def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 제품별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    # ====================================================================
    # [작성]
    result = ...
    # ====================================================================
    # 제품별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        # ====================================================================
        # [작성]
        ...
        # ====================================================================
    return rows

# 검색 후보를 원문 단위로 확인합니다.
seed_documents = vector_documents(query_vector)
display(pd.DataFrame([{**row, "title": documents_by_id[row["source_id"]]["title"]} for row in seed_documents]))


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert seed_documents and len(seed_documents) == len({row["source_id"] for row in seed_documents}), '2번 청크 결과를 원문별로 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert {row["source_id"] for row in seed_documents} == {item.metadata["source_id"] for item in vector_result.items}, '2번 청크 결과를 원문별로 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(row == next(item.metadata for item in vector_result.items if item.metadata["source_id"] == row["source_id"]) for row in seed_documents), '2번 청크 결과를 원문별로 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 2번 핵심 확인 완료!")


## 3. 공통 효능 증상으로 다른 제품까지 확장합니다

**배경**: 기준 제품과 같은 증상을 효능에 적은 다른 제품 문서를 찾습니다.

**제공 코드**: 질문 벡터와 데이터 연결. 확장 Cypher는 이 문항에서 문자열 전체를 작성하며 다음 문항의 검색기에 연결합니다.

**요구사항**:

- <strong><code>expansion_query</code></strong>에 전체 Cypher 문자열을 작성하세요. 검색기가 전달하는 node는 DrugChunk, score는 해당 청크의 유사도입니다. CALL (node) 서브쿼리 안에서 청크별 확장을 처리합니다.
- <strong><code>expansion_query</code></strong>는 node의 FROM_DOCUMENT → DrugDocument → ABOUT_DRUG → 기준 Drug(seed) 경로를 따릅니다. 기준 제품에서 TREATS로 Symptom(symptom)에 갔다가 역방향 TREATS로 다른 Drug(other)를 찾으세요. 양쪽 관계를 first·second로 구분합니다.
- <strong><code>expansion_query</code></strong>에서 other는 seed와 달라야 하며 ABOUT_DRUG로 연결된 DrugDocument(other_doc)가 있어야 합니다. HAS_SIDE_EFFECT는 사용하지 않습니다.
- <strong><code>shared_count</code></strong>는 후보 문서별 count(DISTINCT symptom)이며 $min_shared 이상만 남깁니다. <strong><code>paths</code></strong>에는 seed_title·symptom·first_claim·second_claim 키로 기준 제품명·증상명·양쪽 claim_id를 담은 사전을 중복 없이 모으세요.
- <strong><code>expansion_query</code></strong>의 서브쿼리는 other_doc의 id·title과 paths를 source_id·title·paths로 반환합니다. source_id순 정렬 후 $max_per_seed개로 제한하세요.
- <strong><code>expansion_query</code></strong>의 최종 반환 열은 node.id AS seed_chunk_id, score AS seed_score, source_id, title, paths입니다.

**확인 기준**: 다음 문제에서 관계 방향·공통 증상 수·청크별 확장 한도를 확인합니다.


In [ ]:
# 청크마다 기준 제품의 공통 효능 증상 경로를 찾습니다.
# ====================================================================
# [작성]
expansion_query = ...
# ====================================================================


## 4. VectorCypherRetriever에 확장 쿼리를 연결합니다

**배경**: 벡터 검색 직후 공통 효능 경로를 조회합니다.

**제공 코드**: format_expansion 함수·질문 벡터·metadata 표 출력.

**요구사항**:

- <strong><code>graph_vector_retriever</code></strong>는 VectorCypherRetriever입니다. driver, index_name="drug_chunks", embedder=embedding_model, retrieval_query=expansion_query, result_formatter=format_expansion으로 생성하세요.
- <strong><code>expanded_result</code></strong>에는 graph_vector_retriever.search로 query_vector, top_k=2, query_params={"min_shared": 2, "max_per_seed": 3}을 전달한 결과를 담으세요.

**확인 기준**: 진입 청크마다 최대 3개의 다른 제품 문서와 공통 효능 증상 2개 이상의 경로가 나옵니다.


In [ ]:
def format_expansion(record):
    """후보 제품의 제목과 그 제품을 찾은 청크·효능 경로를 반환합니다."""
    row = record.data()
    return RetrieverResultItem(content=row["title"], metadata=row)


# 벡터 검색 뒤 관계 확장 쿼리를 실행하는 검색기를 만듭니다.
# ====================================================================
# [작성]
graph_vector_retriever = ...
# ====================================================================

# 진입 청크 둘에서 공통 증상 두 개 이상인 제품을 찾습니다.
# ====================================================================
# [작성]
expanded_result = ...
# ====================================================================
display(pd.DataFrame([item.metadata for item in expanded_result.items]))


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(graph_vector_retriever, VectorCypherRetriever), '4번 VectorCypherRetriever에 확장 쿼리를 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert 0 < len(expanded_result.items) <= 6, '4번 VectorCypherRetriever에 확장 쿼리를 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert len({item.metadata["seed_chunk_id"] for item in expanded_result.items}) <= 2, '4번 VectorCypherRetriever에 확장 쿼리를 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(sum(other.metadata["seed_chunk_id"] == item.metadata["seed_chunk_id"] for other in expanded_result.items) <= 3 for item in expanded_result.items), '4번 VectorCypherRetriever에 확장 쿼리를 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all({"seed_chunk_id", "seed_score", "source_id", "title", "paths"} <= item.metadata.keys() for item in expanded_result.items), '4번 VectorCypherRetriever에 확장 쿼리를 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(len({path["symptom"] for path in item.metadata["paths"]}) >= 2 for item in expanded_result.items), '4번 VectorCypherRetriever에 확장 쿼리를 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(path["seed_title"] != item.metadata["title"] for item in expanded_result.items for path in item.metadata["paths"]), '4번 VectorCypherRetriever에 확장 쿼리를 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(any(r["claim_id"] == path[key] and r["relation"] == "TREATS" and nodes_by_id[r["subject_id"]]["name"] == title and nodes_by_id[r["object_id"]]["name"] == path["symptom"] for r in drugs["relations"]) for item in expanded_result.items for path in item.metadata["paths"] for key, title in [("first_claim", path["seed_title"]), ("second_claim", item.metadata["title"])]), '4번 VectorCypherRetriever에 확장 쿼리를 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 4번 핵심 확인 완료!")


## 5. 중복 제품의 효능 경로를 보존합니다

**배경**: 여러 청크에서 같은 제품에 도착해도 서로 다른 공통 증상 근거는 남깁니다.

**제공 코드**: 함수 틀, 입력 결과 순회, source_id별 사전 생성, 경로 순회, 함수 호출과 표 출력·최종 정렬.

**요구사항**:

- <strong><code>target[&quot;seed_score&quot;]</code></strong>에 기존 점수와 row["seed_score"]의 최댓값을 남기세요.
- <strong><code>target[&quot;seed_chunks&quot;]</code></strong>에 row["seed_chunk_id"]가 없을 때만 추가하는 조건문과 갱신문을 작성하세요.
- <strong><code>target[&quot;paths&quot;]</code></strong>에 현재 path가 없을 때만 추가하세요. 동일한 경로만 제외하고 서로 다른 효능 경로는 보존합니다.

**확인 기준**: 원문 ID는 하나씩 남고 원래 청크·효능 경로는 빠지지 않습니다.


In [ ]:
def merge_expansions(result):
    """중복 문서를 합치되 서로 다른 효능 경로는 모두 남깁니다."""
    merged = {}
    for item in result.items:
        row = item.metadata
        source_id = row["source_id"]
        if source_id not in documents_by_id:
            raise ValueError('검색 결과의 원문 ID가 현재 자료에 없습니다. 커널을 재시작하고 적재·검색기 생성·검색 셀을 순서대로 다시 실행하세요.')
        # 같은 원문 ID의 결과를 한곳에 모아 출발 청크와 효능 경로를 기록합니다.
        target = merged.setdefault(source_id, {
            "source_id": source_id, "title": row["title"],
            "seed_score": row["seed_score"], "seed_chunks": [], "paths": [],
        })
        # 같은 후보를 찾은 청크가 여럿이면 가장 높은 유사도를 남깁니다.
        # ====================================================================
        # [작성]
        target["seed_score"] = ...
        # ====================================================================
        # 같은 후보를 찾은 서로 다른 시작 청크를 모두 기록합니다.
        # ====================================================================
        # [작성]
        ...
        # ====================================================================
        # 같은 경로는 한 번만 남기고, 다른 증상의 효능 경로는 보존합니다.
        for path in row["paths"]:
            # ====================================================================
            # [작성]
            ...
            # ====================================================================
    # seed_score가 큰 후보부터 정렬하고, 동점이면 원문 ID순으로 정렬합니다.
    return sorted(merged.values(), key=lambda row: (-row["seed_score"], row["source_id"]))

# 같은 원문의 청크와 경로를 모읍니다.
expanded_documents = merge_expansions(expanded_result)
display(pd.DataFrame(expanded_documents))


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert {row["source_id"] for row in expanded_documents} == {item.metadata["source_id"] for item in expanded_result.items}, '5번 중복 제품의 효능 경로를 보존합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert len(expanded_documents) == len({row["source_id"] for row in expanded_documents}), '5번 중복 제품의 효능 경로를 보존합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert expanded_documents == sorted(expanded_documents, key=lambda row: (-row["seed_score"], row["source_id"])), '5번 중복 제품의 효능 경로를 보존합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(row["seed_score"] == max(item.metadata["seed_score"] for item in expanded_result.items if item.metadata["source_id"] == row["source_id"]) for row in expanded_documents), '5번 중복 제품의 효능 경로를 보존합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(len(row["seed_chunks"]) == len(set(row["seed_chunks"])) and set(row["seed_chunks"]) == {item.metadata["seed_chunk_id"] for item in expanded_result.items if item.metadata["source_id"] == row["source_id"]} for row in expanded_documents), '5번 중복 제품의 효능 경로를 보존합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all({tuple(sorted(path.items())) for path in row["paths"]} == {tuple(sorted(path.items())) for item in expanded_result.items if item.metadata["source_id"] == row["source_id"] for path in item.metadata["paths"]} and len(row["paths"]) == len({tuple(sorted(path.items())) for path in row["paths"]}) for row in expanded_documents), '5번 중복 제품의 효능 경로를 보존합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 5번 핵심 확인 완료!")


## 6. 후보의 효능 경로와 원문을 확인합니다

**배경**: 어떤 기준 제품과 공통 증상으로 후보를 찾았는지 설명합니다.

**제공 코드**: 첫 후보 선택, 결과 변수명, 경로 표와 원문 출력, 기존 판단 문항.

**요구사항**:

- <strong><code>path_rows</code></strong>는 selected_document["paths"]의 각 사전을 복사한 뒤 candidate_title에 selected_document["title"]을 추가한 목록입니다. 기존 순서와 키를 유지하세요.
- <strong><code>selected_text</code></strong>에는 documents_by_id에서 selected_document["source_id"]로 찾은 원문의 text 전체를 담으세요.

**확인 기준**: 공통 증상과 두 효능 관계 ID가 남고 원문도 확인할 수 있습니다.


In [ ]:
# 관계로 선정한 이유와 실제 원문을 대조합니다.
selected_document = expanded_documents[0]
# ====================================================================
# [작성]
path_rows = ...
# ====================================================================
# ====================================================================
# [작성]
selected_text = ...
# ====================================================================
display(pd.DataFrame(path_rows))
print(selected_document["title"], selected_text)


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert selected_document == expanded_documents[0], '6번 후보의 효능 경로와 원문을 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert path_rows == [{**path, "candidate_title": selected_document["title"]} for path in selected_document["paths"]], '6번 후보의 효능 경로와 원문을 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert selected_text == documents_by_id[selected_document["source_id"]]["text"], '6번 후보의 효능 경로와 원문을 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 6번 핵심 확인 완료!")


**판단**: 한 경로의 제품 두 개·공통 증상·양쪽 claim_id를 적고, 후보의 효능도 최초 질문과 맞는지 설명하세요.


*(여기에 판단과 근거를 서술하세요)*


## 7. Jev 언어 판단을 전문 검색에 연결합니다

**배경**: 같은 질문을 한국어 원문에 대한 단어 일치 검색으로 처리합니다.

**제공 코드**: Jev 모델·Choice 설정·KoreanKeywords 스키마·검색어 체인·함수 틀·언어 선택값 읽기·문서 중복 제거·결과 출력.

**요구사항**:

- <strong><code>decision</code></strong>에는 language_classifier.invoke로 {"state": query, "questions": {"language": language_question}}을 전달한 결과를 담으세요.
- <strong><code>search_text</code></strong>를 만드는 조건문 전체를 작성하세요. language가 "en"이면 lexical_chain.invoke에 {"question": query}를 전달해 terms를 받습니다. terms.keywords의 각 단어를 큰따옴표로 감싸 " OR "로 연결하세요. 그 외에는 query를 그대로 search_text에 담습니다.
- <strong><code>hits</code></strong>에는 run_cypher로 실행한 전문 검색 결과를 담으세요. Cypher는 CALL db.index.fulltext.queryNodes('drug_fulltext', $search_text, {limit: $top_k}) YIELD node, score를 사용합니다.
- <strong><code>hits</code></strong>의 반환 열은 node.source_id AS source_id, node.id AS chunk_id, score입니다. score 내림차순·동점 chunk_id순으로 정렬하고 run_cypher에 search_text=search_text·top_k=top_k를 전달하세요.

**확인 기준**: 한글 원문에는 한글로 검색하며 결과의 원문 ID가 중복되지 않습니다.


In [ ]:
# 한국어 원문이므로 Jev의 영어 판단일 때만 검색어를 한국어로 바꿉니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영문 약품명이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


class KoreanKeywords(BaseModel):
    """한국어 원문에서 찾을 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 한국어 핵심어 3~6개")


# 영어 문장에 있는 내용만 한국어 검색어로 변환합니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "영어로 작성된 의약품 문서 검색 질문을 한국어 핵심어 3~6개로 바꾸세요. "
               "각 항목에는 핵심어 하나만 담고 중복을 제거하세요. "
               "질문에 없는 제품명·성분·효능을 추측해 추가하지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(KoreanKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 한국어 전문 검색 후 원문별 최고 점수 청크를 반환합니다."""
    # ====================================================================
    # [작성]
    decision = ...
    # ====================================================================
    language = decision.choices["language"].choice
    # 한글 입력은 그대로 검색하고 영어만 한국어 검색어로 변환합니다.
    # ====================================================================
    # [작성]
    ...
    # ====================================================================
    print("질문 언어:", language, "| 전문 검색어:", search_text)
    # ====================================================================
    # [작성]
    hits = ...
    # ====================================================================
    rows, seen = [], set()
    # 청크 수가 많은 제품이 문서 후보를 여러 번 차지하지 않게 합니다.
    for row in hits:
        # 다른 자료의 ID나 누락된 ID를 결과에 섞지 않습니다.
        if row["source_id"] not in documents_by_id:
            raise ValueError('검색 결과의 원문 ID가 현재 자료에 없습니다. 커널을 재시작하고 적재·검색기 생성·검색 셀을 순서대로 다시 실행하세요.')
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(row)
    return rows

# 벡터 검색과 같은 효능 질문을 사용합니다.
fulltext_rows = fulltext_documents(semantic_query)
display(pd.DataFrame(fulltext_rows))


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(language_classifier, TypeSafeClassifier), "Jev 언어 판별기를 생성하세요."
assert language_classifier.model == os.getenv("TYPESAFE_MODEL", "jev-1.13.0"), "Jev 모델 설정을 확인하세요."
assert isinstance(language_question, Choice) and set(language_question.criteria) == {"ko", "en"}, "ko·en 선택 기준을 확인하세요."
assert KoreanKeywords.model_fields["keywords"].annotation == list[str], "keywords는 문자열 목록입니다."
assert callable(fulltext_documents), "언어 판단과 전문 검색 함수를 작성하세요."

assert 0 < len(fulltext_rows) <= 12, '7번 Jev 언어 판단을 전문 검색에 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert len(fulltext_rows) == len({row["source_id"] for row in fulltext_rows}), '7번 Jev 언어 판단을 전문 검색에 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all({"source_id", "chunk_id", "score"} <= row.keys() and row["source_id"] in documents_by_id for row in fulltext_rows), '7번 Jev 언어 판단을 전문 검색에 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert [row["score"] for row in fulltext_rows] == sorted([row["score"] for row in fulltext_rows], reverse=True), '7번 Jev 언어 판단을 전문 검색에 연결합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 7번 핵심 확인 완료!")


## 8. 검색 방식별로 찾은 원문을 비교합니다

**배경**: 단어 일치와 의미 유사도 검색이 같은 문서와 다른 문서를 찾는지 확인합니다.

**제공 코드**: 두 검색 결과의 source_id 집합 생성, 집합 출력, 각 첫 원문의 제목·본문 출력, 기존 판단 문항.

**요구사항**:

- <strong><code>common_ids</code></strong>에는 두 검색 결과의 교집합을 담으세요.
- <strong><code>fulltext_only</code></strong>에는 전문 검색에만 있는 ID를, <strong><code>dense_only</code></strong>에는 벡터 검색에만 있는 ID를 담으세요.

**확인 기준**: 동일한 원문 ID로 공통·차이 집합을 비교합니다.


In [ ]:
# 같은 원문을 기준으로 검색 결과를 비교합니다.
dense_ids = {row["source_id"] for row in seed_documents}
fulltext_ids = {row["source_id"] for row in fulltext_rows}
# ====================================================================
# [작성]
common_ids = ...
# ====================================================================
# ====================================================================
# [작성]
fulltext_only = ...
# ====================================================================
# ====================================================================
# [작성]
dense_only = ...
# ====================================================================
print("공통:", common_ids, "전문만:", fulltext_only, "벡터만:", dense_only)
for label, rows in [("벡터", seed_documents), ("전문", fulltext_rows)]:
    doc = documents_by_id[rows[0]["source_id"]]
    print(label, doc["title"], doc["text"])


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert dense_ids == {row["source_id"] for row in seed_documents}, '8번 검색 방식별로 찾은 원문을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert fulltext_ids == {row["source_id"] for row in fulltext_rows}, '8번 검색 방식별로 찾은 원문을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert common_ids == dense_ids & fulltext_ids, '8번 검색 방식별로 찾은 원문을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert fulltext_only == fulltext_ids - dense_ids and dense_only == dense_ids - fulltext_ids, '8번 검색 방식별로 찾은 원문을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 8번 핵심 확인 완료!")


**판단**: 각 방식의 첫 문서에서 위산과다·속쓰림·신트림이 어떤 절에 적혀 있는지 확인하고 검색 결과가 질문에 맞는지 적으세요.


*(여기에 판단과 근거를 서술하세요)*
